# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a step-by-step guide for loading and exploring the FAIR^2 dataset using the `mlcroissant` library.

### Dataset Source
The dataset is defined via a Croissant schema, available at:
https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json

In [ ]:
# Ensure mlcroissant is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"{metadata.name}: {metadata.description}")

### Metadata at a glance
- **@id:** https://api.app.sen.science/frontiers/7154140/8875020f-242a-4f80-afd4-a4cbac8715f0
- **Version:** 1.0
- **Keywords:** protein abundance, modification analysis, peptide sequences, molecular weight, human mast cells, extracellular vesicles

**Below, we explore the dataset's structure through its available Record Sets and Fields.**

## 2. Data Overview
List available record sets and fields, referencing everything by `@id`.
This will help identify how the data is organized, which tables exist, and the available features per table.

In [ ]:
# Inspect all available record sets with their @id
record_sets = []
for rs in dataset.metadata.record_sets:
    print(f"Record Set: {rs.id}")
    record_sets.append(rs.id)
    print(f"  Name: {rs.name}")
    print(f"  Description: {rs.description}")
    print("  Fields (by @id):")
    for f in rs.fields:
        print(f"    - {f.id} (name: {f.name}, dataType: {f.data_type})")
    print()

**Note:** For the next steps, pick the primary record set (`@id`) you wish to analyze. Most datasets have at least one main record set containing tabular data.

> We'll use the `@id` for all fields, columns, and tables below.

## 3. Data Extraction
Load records from a record set (i.e., table) into a pandas DataFrame using its `@id`.

> **Tip:** You can use the overview above to decide which record set `@id` to use. Replace `<record_set_id>` with the main one for this dataset.

In [ ]:
# Example: Let's extract all dataframes for all record sets.
dataframes = {}
for record_set_id in record_sets:
    print(f"Loading records for record set @id: {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"  Fields (@id): {df.columns.tolist()}")
        print(df.head())
    else:
        print(f"  No records found for record set: {record_set_id}")
    print()

For the rest of the analysis, we'll focus on the first record set found (`main_record_set_id`). Feel free to change this variable as needed.

In [ ]:
# Pick the main record set for further exploration
main_record_set_id = record_sets[0] if record_sets else None
if main_record_set_id:
    df = dataframes[main_record_set_id]
    print(f"Main DataFrame for record set @id '{main_record_set_id}', columns:")
    print(df.columns.tolist())
    print(df.head())
else:
    print("No record sets with records found in the dataset.")

## 4. Exploratory Data Analysis (EDA)
Let's process and analyze the data from the main record set. All operations reference columns by their `@id` only (not names or indices directly).
We'll:
- Filter records using a numeric field `@id`
- Normalize a numeric field
- Optionally, group by a categorical field

> Replace these IDs with the actual referenced ids found in the overview section for optimal results.

In [ ]:
# Choose a numeric field `@id` from your record set (see above)
# Example: Suppose the field accession is 'accession', and normalized abundance is '@id': 'normalized_abundance'

# Fallback: choose any numeric field present in df.columns as default
import numpy as np
numeric_fields = [col for col in df.columns if pd.api.types.is_numeric_dtype(df[col])]
if numeric_fields:
    numeric_field_id = numeric_fields[0]
else:
    # Otherwise, attempt to typecast possible numeric columns
    for col in df.columns:
        try:
            df[col] = pd.to_numeric(df[col])
            numeric_field_id = col
            break
        except Exception:
            continue
    else:
        raise RuntimeError('No numeric fields available in the dataset!')

print(f"Numeric field chosen (@id): {numeric_field_id}")

# Filter: keep only records where the value is above a threshold (e.g., > 10)
threshold = 10
filtered_df = df[df[numeric_field_id] > threshold]
print(f"Number of records after filtering {numeric_field_id} > {threshold}: {len(filtered_df)}")
print(filtered_df.head())

# Normalize the field
normalized_col = f"{numeric_field_id}_normalized"
filtered_df[normalized_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
print(f"Normalized column '{normalized_col}':")
print(filtered_df[[numeric_field_id, normalized_col]].head())

# Group by another column if desired (choose a non-numeric / categorical one by @id)
categorical_fields = [col for col in df.columns if pd.api.types.is_object_dtype(df[col]) and col != numeric_field_id]
if categorical_fields:
    group_field_id = categorical_fields[0]
    print(f"Grouping by categorical field (@id): {group_field_id}")
    grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
    print(grouped_df.head())
else:
    print("No categorical fields found to group by.")

## 5. Visualization
Visualize numeric distributions or relationships using the selected fields by their `@id`.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Distribution of the numeric field
plt.figure(figsize=(8,4))
sns.histplot(filtered_df[numeric_field_id].dropna(), kde=True)
plt.title(f"Distribution of {numeric_field_id}")
plt.xlabel(numeric_field_id)
plt.ylabel('Count')
plt.show()

# Boxplot grouped by the categorical field if available
if categorical_fields:
    plt.figure(figsize=(10,5))
    sns.boxplot(x=group_field_id, y=numeric_field_id, data=filtered_df)
    plt.title(f"{numeric_field_id} by {group_field_id}")
    plt.xticks(rotation=45)
    plt.show()

## 6. Conclusion
In this notebook, we:
- Loaded metadata and records programmatically from the dataset Croissant schema.
- Identified tables (record sets) and their features (fields) by `@id`.
- Extracted and analyzed data from a main record set, performing filtering, normalization, and grouping using only Croissant `@id` references.
- Generated basic visualizations of numeric data fields.

**Continue exploring combinations of fields by their `@id` to generate additional insights!**

---
**References:**
- [mlcroissant documentation](https://mlcommons.github.io/croissant/api/)
- [FAIR^2 Croissant schema](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json)